# Version Management & Release Operations Guide

**Complete guide for semantic versioning, dependency management, release cycles, and rollback procedures.**

Learn how to manage releases safely and maintain backward compatibility.


## Semantic Versioning Strategy

### Version Format

```
MAJOR.MINOR.PATCH[-PRERELEASE][+BUILD]
 ↑      ↑     ↑        ↑            ↑
 |      |     |        |            └─ Build metadata (ignored in precedence)
 |      |     |        └─ Pre-release (alpha, beta, rc)
 |      |     └─ Bug fixes
 |      └─ New features (backward compatible)
 └─ Breaking changes

Examples:
1.0.0       Production release
1.2.3       Three releases after 1.0.0
2.0.0       Major breaking changes
2.0.0-rc.1  Release candidate
2.1.0-beta  Beta pre-release
```

### Versioning Rules

```python
# pyproject.toml or package.json
version = "1.2.3"

# Increment rules:
# MAJOR: Breaking API changes, incompatible upgrades
#   - Removing endpoints
#   - Changing response format
#   - Changing authentication
#   - Modifying database schema (incompatibly)
#
# MINOR: New features, backward compatible
#   - New endpoints
#   - New optional parameters
#   - New features with defaults
#   - Performance improvements
#
# PATCH: Bug fixes, backward compatible
#   - Security patches
#   - Bug fixes
#   - Documentation updates
#   - Non-breaking refactoring
```

### Version Constraints

```yaml
# pyproject.toml - Dependency version pinning
dependencies:
  fastapi = "^0.100.0"      # >=0.100.0, <1.0.0
  pydantic = "~2.0.0"       # >=2.0.0, <2.1.0
  sqlalchemy = ">=1.4,<2.1" # Exact range
  torch = "2.0.1"           # Exact version (pinned)
```

---

## Dependency Management

### Automated Dependency Updates

```python
# scripts/check_dependencies.py
import subprocess
import json
from packaging import version

class DependencyChecker:
    def __init__(self):
        self.outdated = []
        self.vulnerabilities = []

    def check_for_updates(self):
        """Check for outdated packages."""
        result = subprocess.run(
            ["pip", "list", "--outdated", "--format", "json"],
            capture_output=True,
            text=True
        )
        self.outdated = json.loads(result.stdout)

        return self._classify_updates()

    def _classify_updates(self) -> dict:
        """Classify updates by type (major, minor, patch)."""
        major = []
        minor = []
        patch = []

        for pkg in self.outdated:
            current = version.parse(pkg['version'])
            latest = version.parse(pkg['latest_version'])

            if current.major != latest.major:
                major.append(pkg)
            elif current.minor != latest.minor:
                minor.append(pkg)
            else:
                patch.append(pkg)

        return {
            "major": major,
            "minor": minor,
            "patch": patch
        }

    def check_vulnerabilities(self):
        """Check for known CVEs."""
        result = subprocess.run(
            ["pip-audit", "--desc", "--format", "json"],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:
            self.vulnerabilities = json.loads(result.stdout).get("vulnerabilities", [])

        return self.vulnerabilities

    def generate_report(self) -> str:
        """Generate dependency health report."""
        updates = self.check_for_updates()
        vulns = self.check_vulnerabilities()

        report = f"""
# Dependency Health Report

## Updates Available
- Major: {len(updates['major'])} packages
- Minor: {len(updates['minor'])} packages
- Patch: {len(updates['patch'])} packages

## Vulnerabilities
- Critical: {len([v for v in vulns if v.get('severity') == 'critical'])}
- High: {len([v for v in vulns if v.get('severity') == 'high'])}
- Medium: {len([v for v in vulns if v.get('severity') == 'medium'])}

## Priority Actions
1. Fix all critical/high vulnerabilities (patch releases)
2. Review major versions (breaking changes)
3. Update minor versions (new features)
4. Update patch versions (bug fixes)
"""
        return report
```

### Dependency Pinning Strategy

```yaml
# requirements.txt - Two-tier approach
# Tier 1: Core (pinned for stability)
torch==2.0.1          # ML framework - pinned for reproducibility
transformers==4.30.0  # Model architectures - pinned

# Tier 2: Infrastructure (minor constraint)
fastapi~=0.100.0      # Web framework
sqlalchemy~=2.0.0     # Database ORM

# Tier 3: Utilities (flexible)
pandas>=1.0.0         # Data processing
numpy>=1.20.0         # Numerical computing

# Development only
pytest>=7.0.0
black~=23.0           # Code formatting
```


## Release Management

### Release Process

```bash
#!/bin/bash
# scripts/release.sh

VERSION=$1
if [[ ! $VERSION =~ ^[0-9]+\.[0-9]+\.[0-9]+$ ]]; then
    echo "Usage: ./release.sh <version> (e.g., 1.2.3)"
    exit 1
fi

set -e

echo "🔄 Starting release process for v$VERSION..."

# Step 1: Validate tests pass
echo "✓ Running tests..."
python -m pytest tests/ -v --cov

# Step 2: Update version numbers
echo "✓ Updating version numbers..."
sed -i "s/version = .*/version = \"$VERSION\"/" pyproject.toml
sed -i "s/\"version\": .*/\"version\": \"$VERSION\",/" package.json

# Step 3: Generate changelog
echo "✓ Generating CHANGELOG entry..."
./scripts/generate_changelog.py $VERSION

# Step 4: Commit and tag
echo "✓ Creating git tag..."
git add -A
git commit -m "Release v$VERSION"
git tag -a "v$VERSION" -m "Release version $VERSION"

# Step 5: Push to repository
echo "✓ Pushing to repository..."
git push origin main
git push origin "v$VERSION"

# Step 6: Build and publish
echo "✓ Building distribution packages..."
python -m build

echo "✓ Publishing to package repositories..."
python -m twine upload dist/*

# Step 7: Create GitHub release
echo "✓ Creating GitHub release..."
gh release create "v$VERSION" --generate-notes

echo "✅ Release v$VERSION complete!"
```

### Changelog Format

```markdown
# Changelog

## [1.2.0] - 2026-07-05

### Added

- New chat streaming endpoint with SSE support
- Real-time user presence indicators
- Custom instruction support for Pro tier

### Changed

- Improved semantic memory search performance (+40%)
- Updated authentication to use JWT tokens
- Refactored database connection pooling

### Deprecated

- Old `/api/complete` endpoint (use `/api/chat` instead)

### Removed

- Support for Python 3.9 (minimum now 3.10)

### Fixed

- Fixed memory leak in long-running connections
- Fixed RTL text rendering in UI
- Fixed timezone handling in timestamps

### Security

- Updated OpenSSL to 1.1.1v (CVE-2023-2650)
- Added rate limiting to auth endpoints
```

### Release Checklist

```yaml
# Pre-release
- [ ] All tests passing (unit, integration, E2E)
- [ ] Code review approved
- [ ] Performance benchmarks run
- [ ] Security scan passing (no critical/high)
- [ ] Dependency vulnerabilities addressed
- [ ] Changelog updated
- [ ] Version numbers updated
- [ ] Documentation updated

# Release
- [ ] Git tag created
- [ ] GitHub release created
- [ ] Package published to PyPI/npm
- [ ] Docker image built and pushed
- [ ] Azure deployment prepared

# Post-release
- [ ] Monitor error rates (first 1 hour)
- [ ] Verify all endpoints responding
- [ ] Confirm user traffic normal
- [ ] Update status page
- [ ] Notify support/sales team
```

---

## Managing Breaking Changes

### Deprecation Strategy

```python
# shared/deprecation.py
import warnings
from functools import wraps
from datetime import datetime

def deprecated(removal_version: str, alternative: str = None):
    """Mark function as deprecated."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            message = (
                f"{func.__name__} is deprecated and will be removed in {removal_version}. "
            )
            if alternative:
                message += f"Use {alternative} instead."

            warnings.warn(message, DeprecationWarning, stacklevel=2)
            return func(*args, **kwargs)
        return wrapper
    return decorator

# Usage
@deprecated(removal_version="2.0.0", alternative="chat_completions_v2()")
def chat_completions(messages):
    """Old endpoint - use chat_completions_v2 instead."""
    return process_messages(messages)

# API response with deprecation headers
@app.route("/api/complete", methods=["POST"])
def deprecated_endpoint():
    response = func.HttpResponse(json.dumps({"data": "result"}))
    response.headers["Deprecation"] = "true"
    response.headers["Sunset"] = "Sat, 05 Jan 2027 00:00:00 GMT"
    response.headers["Link"] = '</api/chat>; rel="successor-version"'
    return response
```

### Backward Compatibility

```python
# Maintaining multiple API versions
@app.route("/api/v1/chat", methods=["POST"])
async def chat_v1(req: func.HttpRequest):
    """Legacy API v1 - deprecated."""
    data = req.get_json()
    # Convert v1 format to v2 internally
    v2_data = convert_v1_to_v2(data)
    response = await chat_v2_logic(v2_data)
    # Convert response back to v1 format
    return convert_v2_to_v1(response)

@app.route("/api/v2/chat", methods=["POST"])
async def chat_v2(req: func.HttpRequest):
    """Current API v2."""
    data = req.get_json()
    return await chat_v2_logic(data)

def convert_v1_to_v2(v1_data: dict) -> dict:
    """Convert from v1 request format to v2."""
    return {
        "messages": v1_data.get("text", []),
        "model": v1_data.get("model", "gpt-3.5"),
        "temperature": v1_data.get("temp", 0.7)
    }

def convert_v2_to_v1(v2_response: dict) -> dict:
    """Convert from v2 response format to v1."""
    return {
        "result": v2_response.get("content"),
        "usage": v2_response.get("usage", {})
    }
```


## Rollback Procedures

### Quick Rollback

```bash
#!/bin/bash
# scripts/rollback.sh

ROLLBACK_VERSION=$1

if [ -z "$ROLLBACK_VERSION" ]; then
    echo "Usage: ./rollback.sh <version> (e.g., 1.1.5)"
    exit 1
fi

echo "⚠️  Rolling back to v$ROLLBACK_VERSION..."

# 1. Verify previous version exists
git tag | grep "v$ROLLBACK_VERSION" || {
    echo "❌ Version v$ROLLBACK_VERSION not found"
    exit 1
}

# 2. Create rollback commit
git revert --no-commit HEAD..v$ROLLBACK_VERSION
git commit -m "Rollback: Revert to v$ROLLBACK_VERSION"

# 3. Tag rollback version
git tag -a "v${ROLLBACK_VERSION}-rollback" -m "Rollback to v$ROLLBACK_VERSION"

# 4. Push changes
git push origin main
git push origin "v${ROLLBACK_VERSION}-rollback"

# 5. Redeploy previous version
az functionapp deployment source config-zip \
    --resource-group aria-rg \
    --name aria-functions \
    --src dist/aria-functions-${ROLLBACK_VERSION}.zip

echo "✅ Rollback to v$ROLLBACK_VERSION complete"
echo "⏱️  Monitoring error rates for next 10 minutes..."

# 6. Monitor
for i in {1..10}; do
    sleep 60
    error_rate=$(curl -s http://aria-api/api/ai/status | jq '.error_rate_1m')
    echo "[$i/10] Error rate: $error_rate%"
done
```

### Database Rollback

```python
# scripts/db_rollback.py
import psycopg2
from datetime import datetime, timedelta

class DatabaseRollback:
    def __init__(self, connection_string: str):
        self.conn = psycopg2.connect(connection_string)

    def list_backups(self) -> list:
        """List available database backups."""
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT backup_id, created_at, size_mb, status
            FROM backup_history
            ORDER BY created_at DESC
            LIMIT 10
        """)
        return cursor.fetchall()

    def restore_from_backup(self, backup_id: str):
        """Restore database from backup."""
        cursor = self.conn.cursor()

        # Verify backup exists
        cursor.execute(
            "SELECT * FROM backup_history WHERE backup_id = %s",
            (backup_id,)
        )
        backup = cursor.fetchone()

        if not backup:
            raise ValueError(f"Backup {backup_id} not found")

        # Start restore process
        print(f"Restoring from backup {backup_id}...")

        # Use Azure recovery point
        restore_cmd = f"""
        az sql server backup restore \
            --resource-group aria-rg \
            --server aria-sql \
            --database aria-db \
            --backup-id {backup_id}
        """

        # Execute restore
        os.system(restore_cmd)

        # Verify restore
        cursor.execute("SELECT COUNT(*) FROM users")
        count = cursor.fetchone()[0]
        print(f"✅ Restore complete. {count} users found.")
```


## Version Management Best Practices

### Do's

- ✓ Use semantic versioning consistently
- ✓ Tag all releases in git
- ✓ Write detailed changelogs
- ✓ Pin critical dependencies
- ✓ Deprecate APIs before removing
- ✓ Maintain multiple API versions
- ✓ Test upgrades before releasing
- ✓ Automate dependency checks
- ✓ Plan breaking changes in major versions
- ✓ Use pre-release versions for testing

### Don'ts

- ✗ Skip version numbering
- ✗ Make undocumented breaking changes
- ✗ Skip testing before release
- ✗ Pin all dependencies (too rigid)
- ✗ Break APIs without deprecation period
- ✗ Release without changelog
- ✗ Ignore dependency vulnerabilities
- ✗ Forget to update documentation
- ✗ Don't communicate breaking changes
- ✗ Release too frequently (stability > speed)

### Release Cadence

- **Patch releases**: As needed (security/bug fixes)
- **Minor releases**: Monthly (new features)
- **Major releases**: Quarterly or annually (breaking changes)
- **Pre-releases**: Continuous (alpha/beta testing)
